# Annotation coverage check

Compare the **filtered** source CSVs against the **annotated/labelled** CSVs for all three models
(`GPT-5.2`, `Gemini 3.5 flash`, `Sonet 4.6`).

For every filtered file the notebook finds the matching labelled file in each model folder and checks:

1. **No comment is lost** — every filtered comment appears in the labelled file (extras/duplicates flagged too).
2. **Annotations are present** — the 4 label columns exist and are non-empty.
3. **Annotations are valid** — each label column only uses allowed values:

| column | allowed values |
|---|---|
| `intent` | Pro-Migration, Anti-Migration, Trapped/Regretful, Neutral/Observation |
| `primary_driver` | Economic Necessity, Family Obligation, Systemic/Political Anger, Patriotism/Love |
| `value_orientation` | Collectivist-Family, Collectivist-Nation, Individualist-Self |
| `affect` | Despairing/Sad, Angry/Frustrated, Hopeful/Motivated, Pragmatic |

**To use:** set the two paths in the next cell, then *Run All*.

In [ ]:
# ============================================================
#  CONFIG — edit these two paths
# ============================================================

# Folder that contains the filtered source CSVs (with source subfolders).
FILTERED_DIR = "../data/filtered"

# Folder that contains one subfolder per model, each mirroring the filtered layout.
LABEL_DIR = "../data/annotations/label"

# How many example rows to print per issue.
MAX_SHOW = 10

In [ ]:
import csv
import re
from collections import Counter
from pathlib import Path

# Allowed values per annotation column.
ALLOWED = {
    "intent": {
        "Pro-Migration", "Anti-Migration", "Trapped/Regretful", "Neutral/Observation",
    },
    "primary_driver": {
        "Economic Necessity", "Family Obligation",
        "Systemic/Political Anger", "Patriotism/Love",
    },
    "value_orientation": {
        "Collectivist-Family", "Collectivist-Nation", "Individualist-Self",
    },
    "affect": {
        "Despairing/Sad", "Angry/Frustrated", "Hopeful/Motivated", "Pragmatic",
    },
}
LABEL_COLS = list(ALLOWED.keys())

# Possible names of the comment column in the *filtered* files.
FILTERED_COMMENT_COLS = ["Comment", "Text", "comment"]
# Comment column name in the *labelled* files.
LABEL_COMMENT_COL = "comment"

_WS = re.compile(r"\s+")
_BR = re.compile(r"<br\s*/?>", re.IGNORECASE)

In [ ]:
def normalise(text):
    """Normalise comment text so trivial formatting diffs don't count as loss."""
    if text is None:
        return ""
    text = _BR.sub(" ", text)
    text = text.replace("​", "").strip()
    text = _WS.sub(" ", text)
    return text.lower()


def read_csv(path):
    with Path(path).open(newline="", encoding="utf-8-sig") as fh:
        reader = csv.DictReader(fh)
        rows = list(reader)
        return reader.fieldnames or [], rows


def pick_comment_col(fieldnames, candidates):
    for c in candidates:
        if c in fieldnames:
            return c
    lower = {f.lower(): f for f in fieldnames}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    return None


# Filename suffixes that differ between the filtered and labelled copies
# (e.g. ..._Filtered.csv  vs  ..._Annotated.csv  vs  ..._persona.csv).
_SUFFIX = re.compile(r"(_filtered|_annotated|_persona)+$", re.IGNORECASE)


def base_stem(name):
    """Filename stem with the _Filtered/_Annotated/_persona suffix removed."""
    return _SUFFIX.sub("", Path(name).stem).lower()


def find_label_file(model_dir, rel):
    """Find the labelled file matching a filtered file's relative path.

    Matches on the base name, ignoring the _Filtered / _Annotated / _persona
    suffix, so renamed copies still pair up.
    """
    direct = model_dir / rel
    if direct.exists():
        return direct
    target = model_dir / rel.parent
    if not target.is_dir():
        return None
    want = base_stem(rel.name)
    for f in sorted(target.glob("*.csv")):
        if base_stem(f.name) == want:
            return f
    return None

In [ ]:
def check_pair(filtered_path, label_path, max_show=MAX_SHOW, verbose=True):
    """Compare one filtered file against one labelled file. Returns a result dict."""
    issues = []

    f_fields, f_rows = read_csv(filtered_path)
    l_fields, l_rows = read_csv(label_path)

    f_col = pick_comment_col(f_fields, FILTERED_COMMENT_COLS)
    l_col = pick_comment_col(l_fields, [LABEL_COMMENT_COL])

    if f_col is None:
        issues.append(f"filtered file has no comment column (looked for {FILTERED_COMMENT_COLS})")
    if l_col is None:
        issues.append(f"labelled file has no '{LABEL_COMMENT_COL}' column; got {l_fields}")

    n_filtered, n_label = len(f_rows), len(l_rows)

    # --- comment coverage ---
    missing, extra = [], []
    if f_col and l_col:
        f_counts = Counter(normalise(r.get(f_col, "")) for r in f_rows)
        l_counts = Counter(normalise(r.get(l_col, "")) for r in l_rows)
        for text, cnt in f_counts.items():
            have = l_counts.get(text, 0)
            if have < cnt:
                missing.append((text, cnt - have))
        for text, cnt in l_counts.items():
            have = f_counts.get(text, 0)
            if cnt > have:
                extra.append((text, cnt - have))
        if missing:
            issues.append(f"{sum(c for _, c in missing)} filtered comment(s) missing from labels")
        if extra:
            issues.append(f"{sum(c for _, c in extra)} label comment(s) not found in filtered")

    # --- label column presence & validity ---
    invalid = {c: Counter() for c in LABEL_COLS}
    empty = Counter()
    missing_cols = [c for c in LABEL_COLS if c not in l_fields]
    if missing_cols:
        issues.append(f"labelled file missing annotation column(s): {missing_cols}")
    for r in l_rows:
        for c in LABEL_COLS:
            if c not in l_fields:
                continue
            val = (r.get(c) or "").strip()
            if val == "":
                empty[c] += 1
            elif val not in ALLOWED[c]:
                invalid[c][val] += 1
    for c in LABEL_COLS:
        if empty[c]:
            issues.append(f"{empty[c]} empty value(s) in '{c}'")
        if invalid[c]:
            issues.append(f"{sum(invalid[c].values())} invalid value(s) in '{c}'")

    ok = not issues

    if verbose:
        status = "OK" if ok else "ISSUES"
        print(f"  [{status}] {Path(label_path).name}")
        print(f"        filtered rows: {n_filtered}   labelled rows: {n_label}   "
              f"(comment col: filtered='{f_col}', label='{l_col}')")
        for msg in issues:
            print(f"        - {msg}")
        if missing:
            print(f"        missing comments (showing up to {max_show}):")
            for text, cnt in missing[:max_show]:
                tag = f" x{cnt}" if cnt > 1 else ""
                print(f"            \u00b7 {text[:100]!r}{tag}")
        for c in LABEL_COLS:
            if invalid[c]:
                shown = ", ".join(f"{v!r}({n})" for v, n in invalid[c].most_common(max_show))
                print(f"        invalid {c}: {shown}")

    return {
        "model": None,  # filled in by the run loop
        "file": str(Path(filtered_path)),
        "label_file": str(Path(label_path)),
        "n_filtered": n_filtered,
        "n_label": n_label,
        "missing": sum(c for _, c in missing),
        "extra": sum(c for _, c in extra),
        "empty": sum(empty.values()),
        "invalid": sum(sum(v.values()) for v in invalid.values()),
        "ok": ok,
    }

In [ ]:
# ============================================================
#  RUN — detailed per-model, per-file report
# ============================================================

filtered_root = Path(FILTERED_DIR)
label_root = Path(LABEL_DIR)
assert filtered_root.is_dir(), f"filtered dir not found: {filtered_root}"
assert label_root.is_dir(), f"label dir not found: {label_root}"

filtered_files = sorted(filtered_root.rglob("*.csv"))
models = sorted(p for p in label_root.iterdir() if p.is_dir())

print(f"Filtered files: {len(filtered_files)}   Models: {[m.name for m in models]}\n")

summary = []
all_ok = True
for model in models:
    print(f"=== Model: {model.name} ===")
    for fpath in filtered_files:
        rel = fpath.relative_to(filtered_root)
        lpath = find_label_file(model, rel)
        if lpath is None:
            print(f"  [MISSING] no labelled file for {rel}")
            summary.append({"model": model.name, "file": str(rel), "n_filtered": None,
                            "n_label": None, "missing": None, "extra": None,
                            "empty": None, "invalid": None, "ok": False})
            all_ok = False
            continue
        res = check_pair(fpath, lpath, MAX_SHOW, verbose=True)
        res["model"] = model.name
        res["file"] = str(rel)
        summary.append(res)
        all_ok = all_ok and res["ok"]
    print()

print("=" * 60)
print("ALL CHECKS PASSED" if all_ok else "SOME CHECKS FAILED \u2014 see issues above")

In [ ]:
# ============================================================
#  SUMMARY TABLE
# ============================================================
# Uses pandas if available; otherwise prints a plain aligned table.

cols = ["model", "file", "n_filtered", "n_label", "missing", "extra", "empty", "invalid", "ok"]

try:
    import pandas as pd
    df = pd.DataFrame(summary)[cols]
    display(df)
except Exception:
    widths = {c: max([len(c)] + [len(str(r.get(c))) for r in summary]) for c in cols}
    header = "  ".join(c.ljust(widths[c]) for c in cols)
    print(header)
    print("-" * len(header))
    for r in summary:
        print("  ".join(str(r.get(c)).ljust(widths[c]) for c in cols))

In [ ]:
# ============================================================
#  OPTIONAL: inspect one file in detail
# ============================================================
# Set MODEL and FILE (relative path under the filtered dir) to drill into a
# single comparison and list every missing comment.

MODEL = "Sonet 4.6"
FILE = "Thaha Research/Thaha_Research_Filtered.csv"

fpath = filtered_root / FILE
lpath = find_label_file(label_root / MODEL, Path(FILE))
if lpath is None:
    print(f"No labelled file found for {MODEL} / {FILE}")
else:
    check_pair(fpath, lpath, max_show=10**9, verbose=True)